In [2]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import os
import sys
N_up = 3
nb_dir = '/'.join(os.getcwd().split('/')[:-N_up])
if nb_dir not in sys.path:
    sys.path.append(nb_dir)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import numpy as np
import matplotlib.pyplot as plt



In [4]:
cuda = False

# Setup for Tabular

We generate artificial datasets with train/test sets matching the original ones

In [5]:
names = ['wine', 'default_credit', 'compas', 'lsat']

art_Npoints = [1438+160, 27000+3000, 5554+618, 17432+4358]
art_Ntrain = [1438, 27000, 5554, 17432]

test_dims = [[11], [29, 30], [17, 18], [12]] 
test_dims_rel = [-1, [-2, -1], [-2, -1], -1] 

bnn_widths = [200, 200, 200, 200]
bnn_depths = [2, 2, 2, 2]

vae_widths = [300, 300, 300, 300] # [200, 200, 200, 200]
vae_depths = [3, 3, 3, 3] # We go deeper because we are using residual models
vae_latent_dims = [6, 8, 4, 4]

vaeac_widths = [350, 350, 350, 350] # Bigger than VAE because the task of modelling all conditionals is more complex
vaeac_depths = [3, 3, 3, 3] # We go deeper because we are using residual models
vaeac_latent_dims = [6, 8, 4, 4]
vaeac_under_latent_dims = [6, 8, 4, 4] # following the original paper we set dim(u) = dim(z) with d>r [r is true manifold dim]
vaeac_under_latent_dims2 = [4, 6, 3, 3] # following the original paper we set dim(u) = dim(z) with d>r [r is true manifold dim]

# For automatic explainer generation

regression_bools = [True, False, False, True]
gauss_cat_vae_bools = [False, True, True, True]
flat_vae_bools = [False, False, False, False]
flat_vaeac_bools = [False, True, False, False]

var_names = {}
var_names_flat = {}

var_names['wine'] = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide',
            'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
var_names_flat['wine'] = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide',
            'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']

var_names['default_credit'] = ['Given credit', 'Gender', 'Education', 'Marital status', 'Age', 'Payment delay 1', 'Payment delay 2',
            'Payment delay 3', 'Payment delay 4', 'Payment delay 5', 'Payment delay 6', 'Bill statement 1', 'Bill statement 2',
            'Bill statement 3', 'Bill statement 4', 'Bill statement 5', 'Bill statement 6', 'Previous payment 1', 'Previous payment 2',
            'Previous payment 3', 'Previous payment 4', 'Previous payment 5', 'Previous payment 6']
var_names_flat['default_credit'] = ['Given credit', 'Gender M', 'Gender F', 'Education grad', 'Education under', 'Education HS', 'Education Other',
                 'Marital status M', 'Marital status S', 'Marital status Other', 'Age', 'Payment delay 1', 'Payment delay 2',
            'Payment delay 3', 'Payment delay 4', 'Payment delay 5', 'Payment delay 6', 'Bill statement 1', 'Bill statement 2',
            'Bill statement 3', 'Bill statement 4', 'Bill statement 5', 'Bill statement 6', 'Previous payment 1', 'Previous payment 2',
            'Previous payment 3', 'Previous payment 4', 'Previous payment 5', 'Previous payment 6']

var_names['compas'] = ["age_cat", "race", "sex", "c_charge_degree", 'is_recid', 'priors_count', 'time_served']
var_names_flat['compas'] = ['25 - 45', 'Greater than 45', 'Less than 25', 'African-American', 'Asian', 'Caucasian', 'Hispanic', 'Native American', 'Other', 'Female', 'Male', 'Felony', 'misdemeanour', 'not_recid', 'is_recid', 'priors_count', 'time_served']

var_names['lsat'] = ['LSAT', 'UGPA', 'race', 'sex']
var_names_flat['lsat'] = ['LSAT', 'UGPA', 'amerind', 'mexican', 'other', 'black', 'asian', 'puerto', 'hisp', 'white', 'female', 'male']



In [6]:
dname = 'lsat'
d_idx = names.index(dname)

In [7]:
def join_LSAT_targets(x_train, x_test, y_train, y_test, input_dim_vec):
    input_dim_vec = np.append(input_dim_vec, 1)

    x_train = np.concatenate([x_train, y_train], axis=1)
    x_test = np.concatenate([x_test, y_test], axis=1)
    return x_train, x_test, input_dim_vec

In [8]:

from counterfactual_xai.utils.lsat_dataloader import LsatDataloader

x_train, x_test, x_means, x_stds, y_train, y_test, y_means, y_stds, my_data_keys, input_dim_vec = \
LsatDataloader(input_dims=[1, 1, 8, 2], csv_path="./../data/").get_lsat_dataset()
xy_means = np.concatenate([x_means, y_means])
xy_stds = np.concatenate([x_stds, y_stds])
xy_train, xy_test, input_dim_vec_xy = join_LSAT_targets(x_train, x_test, y_train, y_test, input_dim_vec)
print('LSAT', x_train.shape, x_test.shape)
print(input_dim_vec_xy)



LSAT (17432, 12) (4358, 12)
[1 1 8 2 1]


In [9]:
gauss_cat_vae_bools[d_idx]

True

## Load GT VAEAC



In [12]:
from counterfactual_xai.utils.clue.vae.gaussian_vae import VAEACGaussCatNet, under_VAEAC

# LOAD VAEAC

width = vaeac_widths[d_idx]
depth = vaeac_depths[d_idx] # number of hidden layers
latent_dim = vaeac_latent_dims[d_idx]
lr = 3e-4
VAEAC = VAEACGaussCatNet(input_dim_vec_xy, width, depth, latent_dim, pred_sig=False,
                          lr=lr, cuda=cuda, flatten=flat_vaeac_bools[d_idx])
VAEAC.load('../../saves/fc_preact_VAEAC_NEW_' + dname + '_models/theta_best.dat')

# LOAD Under-NET

base_network = VAEAC.model
width = 150
depth = 2
latent_dim = vaeac_under_latent_dims[d_idx] #VAEAC.latent_dim # change to 2 for better generations
lr = 3e-4


under_VAEAC_net = under_VAEAC(base_network, width, depth, latent_dim, lr, cuda=cuda)
under_VAEAC_net.load('../../saves/fc_VAEAC_NEW_under_' + dname + '_models/theta_best.dat')


y VAE_gauss_net
    Total params: 0.77M


TypeError: BaseNet.load() missing 1 required positional argument: 'device'

# Generate art data

In [10]:
from counterfactual_xai.utils.clue.activations import selective_softmax
from counterfactual_xai.utils.clue.evaluation.utils import sample_artificial_dataset, sample_artificial_targets_cat, sample_artificial_targets_gauss
from counterfactual_xai.utils.datafeed import DataFeedCluePaper

Npoints = art_Npoints[d_idx]
Ntrain = art_Ntrain[d_idx]

x_art, y_art, xy_art = sample_artificial_dataset(under_VAEAC_net, test_dims[d_idx], Npoints,
                                u_dims=vaeac_under_latent_dims[d_idx],
                                                 sig=((not gauss_cat_vae_bools[d_idx]) and regression_bools[d_idx]),
                                                 softmax=False)

if gauss_cat_vae_bools[d_idx]:

    x_art = selective_softmax(x_art, input_dim_vec, grad=False, cat_probs=True, prob_sample=True) # makes output probabilistic and draws sample
    xy_art = selective_softmax(xy_art, input_dim_vec_xy, grad=False, cat_probs=True, prob_sample=True)

y_art_gen = y_art

conditional_targets = False 

    
if not regression_bools[d_idx]:
    if conditional_targets:
        y_art = sample_artificial_targets_cat(VAEAC, xy_art, test_dims[d_idx], N_target_samples=500,
                                              z_mean=False, softmax=False).data
        y_art = y_art.mean(dim=0)
    
    else:
        y_art = selective_softmax(y_art, [2], grad=False, cat_probs=True, prob_sample=True)
        
    print(y_art.shape)
    _, y_art_BNN = y_art.max(dim=1)  # get integer label
    
else:
    if conditional_targets:
        y_art, stds = sample_artificial_targets_gauss(VAEAC, xy_art, test_dims[d_idx], N_target_samples=500,
                                        pred_sig=((not gauss_cat_vae_bools[d_idx]) and regression_bools[d_idx]),
                                        z_mean=False)
        y_art = y_art.mean(dim=0)
    
    
    print(y_art.shape)
    y_art_BNN = y_art
    if len(y_art_BNN.shape) == 1:
        y_art_BNN = y_art_BNN.unsqueeze(1)

        
###################################################################################################

# xy_art_train = xy_art[:Ntrain, :].cpu().numpy()
# xy_art_test = xy_art[Ntrain:, :].cpu().numpy()

x_art_train = x_art[:Ntrain, :].cpu().numpy()
x_art_test = x_art[Ntrain:, :].cpu().numpy()

y_art_train = y_art[:Ntrain, :].cpu().numpy()
y_art_test = y_art[Ntrain:, :].cpu().numpy()

if regression_bools[d_idx]:
    y_art_BNN_train = y_art_BNN[:Ntrain, :].cpu().numpy()
    y_art_BNN_test = y_art_BNN[Ntrain:, :].cpu().numpy()
    
else:
    y_art_BNN_train = y_art_BNN[:Ntrain].cpu().numpy()
    y_art_BNN_test = y_art_BNN[Ntrain:].cpu().numpy()
    
print('------------')
print(x_art_train.shape)
print(y_art_train.shape)
print(x_art_test.shape)
print(y_art_test.shape)

art_trainset_BNN = DataFeedCluePaper(x_art_train, y_art_BNN_train, transform=None)
art_valset_BNN = DataFeedCluePaper(x_art_test, y_art_BNN_test, transform=None)

art_trainset = DataFeedCluePaper(x_art_train, y_art_train, transform=None)
art_valset = DataFeedCluePaper(x_art_test, y_art_test, transform=None)

# art_x_trainset = Datafeed(x_art_train, x_art_train, transform=None)
# art_x_valset = Datafeed(x_art_test, x_art_test, transform=None)


torch.Size([21790, 1])
------------
(17432, 12)
(17432, 1)
(4358, 12)
(4358, 1)


# Train models on artificial data

We recomend using pretrained models for faster results

In [13]:
use_pretrained_ART = False

## Train BNN

In [14]:
import torch
from torchvision import datasets, transforms

from counterfactual_xai.utils.clue.bnn.mlp import MLP
from counterfactual_xai.utils.clue.gaussian_mlp import GaussianMLP
from counterfactual_xai.utils.clue.bnn.gaussian_bnn import GaussianBNN, BNNCategorical

from counterfactual_xai.utils.clue.bnn.train_regression import train_BNN_regression
from counterfactual_xai.utils.clue.bnn.train_classification import train_BNN_classification
import numpy as np

if regression_bools[names.index(dname)]:
    
    y_means = torch.Tensor(y_means)
    y_stds = torch.Tensor(y_stds)
    
    input_dim = x_art_train.shape[1]
    width = bnn_widths[names.index(dname)]
    depth = bnn_depths[names.index(dname)]
    output_dim = y_art_train.shape[1]
    model = GaussianMLP(input_dim, width, depth, output_dim, flatten_image=False)

    N_train = x_art_train.shape[0]
    lr = 1e-2
    cuda = torch.cuda.is_available()
    art_BNN = GaussianBNN(model, N_train, lr=lr, cuda=cuda)
    
    batch_size = 512#
    nb_epochs = 2400 # We can do less iterations as this method has faster convergence

    ## weight saving parameters #######
    burn_in = 120 # this is in epochs 
    sim_steps = 20 # We want less correlated samples -> despite having per minibatch noise we see correlations
    N_saves = 100
    resample_its = 10
    resample_prior_its = 50 # 45 can be choosen to better control overfitting 
    re_burn = 1e7
    nb_its_dev = 10

    save_dir = '../../saves/fc_BNN_NEW_ART_' + dname

    if not use_pretrained_ART:
        cost_train, cost_dev, rms_dev, ll_dev = train_BNN_regression(art_BNN, save_dir, batch_size, nb_epochs, art_trainset_BNN, art_valset_BNN, cuda,
                                         burn_in, sim_steps, N_saves, resample_its, resample_prior_its,
                                         re_burn, flat_ims=False, nb_its_dev=nb_its_dev, y_mu=y_means, y_std=y_stds)
    
else:
    N_train = x_art_train.shape[0]
    input_dim = x_art_train.shape[1]
    width = bnn_widths[names.index(dname)]
    depth = bnn_depths[names.index(dname)]
    output_dim = 2
    model = MLP(input_dim, width, depth, output_dim, flatten_image=False)

    lr = 1e-2
    cuda = torch.cuda.is_available()
    art_BNN = BNNCategorical(model, N_train, lr=lr, cuda=cuda)
    
    batch_size = 512#
    nb_epochs = 2400 # We can do less iterations as this method has faster convergence
    
    burn_in = 120 # this is in epochs 
    sim_steps = 20 # We want less correlated samples -> despite having per minibatch noise we see correlations
    N_saves = 100
    resample_its = 10
    resample_prior_its = 50
    re_burn = 1e7
    nb_its_dev = 10
    
    
    save_dir = '../../saves/fc_BNN_NEW_ART_' + dname

    if not use_pretrained_ART:
        cost_train, cost_dev, err_train, err_dev = train_BNN_classification(art_BNN, save_dir, batch_size,
                                 nb_epochs, art_trainset_BNN, art_valset_BNN, cuda,
                                 burn_in, sim_steps, N_saves, resample_its, resample_prior_its,
                                 re_burn, flat_ims=False, nb_its_dev=nb_its_dev)


art_BNN.load_weights(save_dir + '_models/state_dicts.pkl')




Net:
 Creating Net!! 
BNN gaussian output
    Total params: 0.04M

Network:

Train:
  init cost variables:


/Users/lukasscholz/repositorys/studienprojekt/CLUE/BNN/sampler.py:82: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/python_arg_parser.cpp:1630.)
  d_p.add_(weight_decay, p.data)


it 0/2400, Jtr_pred = 1.079316,    time: 19.791283 seconds

    Jdev = 1.034015 

 Loglike = -0.9627, rms = 0.6361

best test error
it 1/2400, Jtr_pred = 1.031764,    time: 19.112250 seconds

it 2/2400, Jtr_pred = 1.021056,    time: 18.816443 seconds

it 3/2400, Jtr_pred = 1.022990,    time: 18.903111 seconds

it 4/2400, Jtr_pred = 1.020862,    time: 24.817953 seconds

it 5/2400, Jtr_pred = 1.024902,    time: 18.876943 seconds

it 6/2400, Jtr_pred = 1.023280,    time: 18.857680 seconds

it 7/2400, Jtr_pred = 1.023913,    time: 18.759133 seconds

it 8/2400, Jtr_pred = 1.018424,    time: 18.771792 seconds

it 9/2400, Jtr_pred = 1.018823,    time: 19.166273 seconds

it 10/2400, Jtr_pred = 1.019608,    time: 18.876872 seconds

    Jdev = 1.028092 

 Loglike = -0.9568, rms = 0.6298

best test error
it 11/2400, Jtr_pred = 1.022915,    time: 18.920597 seconds

it 12/2400, Jtr_pred = 1.021555,    time: 19.538375 seconds

it 13/2400, Jtr_pred = 1.019586,    time: 19.433222 seconds

it 14/2400, 

KeyboardInterrupt: 

## Train VAE



In [ ]:
from counterfactual_xai.utils.clue.vae.gaussian_vae import VAE_gauss_net
import torch
from torchvision import datasets, transforms
# from VAE.fc_gauss import VAE_gauss_net
# from VAE.fc_gauss_cat import VAE_gauss_cat_net

from counterfactual_xai.utils.clue.vae.gaussian_vae import GaussianVAE
from counterfactual_xai.utils.clue.vae.train import train_VAE


width = vae_widths[names.index(dname)]
depth = vae_depths[names.index(dname)] # number of hidden layers
latent_dim = vae_latent_dims[names.index(dname)]

if not gauss_cat_vae_bools[names.index(dname)]:
    input_dim = x_art_train.shape[1]
    lr = 1e-4
    cuda = torch.cuda.is_available()
    VAE_art = VAE_gauss_net(input_dim, width, depth, latent_dim, pred_sig=False, lr=lr, cuda=cuda)
    
    batch_size = 128
    nb_epochs = 2500
    early_stop = 200

    save_dir = '../../saves/fc_preact_VAE_NEW(300)_ART_' + dname
    
    if not use_pretrained_ART:
        vlb_train, vlb_dev = train_VAE(VAE_art, save_dir, batch_size, nb_epochs, art_trainset, art_valset,
                                       cuda=cuda, flat_ims=False, train_plot=False, early_stop=early_stop)
    
else:
    cuda = torch.cuda.is_available()
    lr = 1e-4
    VAE_art = GaussianVAE(input_dim_vec, width, depth, latent_dim, pred_sig=False,
                            lr=lr, cuda=cuda, flatten=flat_vae_bools[names.index(dname)])  
    
    batch_size = 128
    nb_epochs = 2500
    lr = 1e-4
    early_stop = 200


    save_dir = '../../saves/fc_preact_VAE_NEW(300)_ART_' + dname
    
    if not use_pretrained_ART:
        vlb_train, vlb_dev = train_VAE(VAE_art, save_dir, batch_size, nb_epochs, art_trainset, art_valset,
                                       cuda=cuda, flat_ims=False, train_plot=False, early_stop=early_stop)

VAE_art.load(save_dir + '_models/theta_best.dat')


## Train VAEAC noclass 

In [ ]:
from numpy.random import uniform, binomial

class top_masker:
    """
    Returned mask is sampled from component-wise independent Bernoulli
    distribution with probability of component to be unobserved p.
    Such mask induces the type of missingness which is called
    in literature "missing completely at random" (MCAR).
    If some value in batch is missed, it automatically becomes unobserved.
    """
    def __init__(self, p):
        self.p = p

    def __call__(self, batch):
        pp = uniform(low=0.0, high=self.p, size=batch.shape[0])
        pp = np.expand_dims(pp, axis=1)
        pp = np.repeat(pp, batch.shape[1], axis=1)
        nan_mask = torch.isnan(batch).float()  # missed values
#         bernoulli_mask_numpy = np.random.choice(2, size=batch.shape,
#                                                 p=[1 - pp, pp])
        bernoulli_mask_numpy = binomial(1, pp, size=None)
#         print(bernoulli_mask_numpy.shape)
        bernoulli_mask = torch.from_numpy(bernoulli_mask_numpy).float()
        mask = torch.max(bernoulli_mask, nan_mask)  # logical or
        return mask


In [ ]:
from counterfactual_xai.utils.clue.vae.train_vaec import train_VAEAC


masker = top_masker(p=1)

width = vaeac_widths[d_idx]
depth = vaeac_depths[d_idx] # number of hidden layers
print(input_dim, width, depth)
latent_dim = vaeac_latent_dims[d_idx]
cuda = torch.cuda.is_available()

if not gauss_cat_vae_bools[d_idx]:

    input_dim = x_art_train.shape[1]

    VAEAC_art = VAE_gauss_net(input_dim, width, depth, latent_dim, pred_sig=False, lr=lr, cuda=cuda)
    
    batch_size = 128
    nb_epochs = 2500
    lr = 1e-4
    early_stop = 200
    
    save_dir = '../../saves/fc_preact_VAEAC_NEW_ART_' + dname
    
    if not use_pretrained_ART:
        vlb_train, vlb_dev = train_VAEAC(VAEAC_art, masker, save_dir, batch_size, nb_epochs, art_trainset, art_valset, cuda,
                    flat_ims=False, train_plot=False, Nclass=None, early_stop=early_stop)
    
else:
    batch_size = 128
    nb_epochs = 2500
    lr = 1e-4
    early_stop = 200

    VAEAC_art = GaussianVAE(input_dim_vec, width, depth, latent_dim, pred_sig=False,
                                lr=lr, cuda=cuda, flatten=False)
    
    save_dir = '../../saves/fc_preact_VAEAC_NEW_ART_' + dname
    
    if not use_pretrained_ART:
        vlb_train, vlb_dev = train_VAEAC(VAEAC_art, masker, save_dir, batch_size, nb_epochs, art_trainset, art_valset, cuda,
                    flat_ims=False, train_plot=False, Nclass=None, early_stop=early_stop)

VAEAC_art.load(save_dir + '_models/theta_best.dat')


# Run experiments

In [ ]:

from os import mkdir

experiment_dir = '../../experiment_data/NEW_art_' + dname + '/'
print(experiment_dir)
mkdir(experiment_dir)

### do some thresholding

We specify different thresholds for epistemic and aleatoric uncertainty. These are different from the ones for non artificial data as the g.t. VAEAC changes dataset statistics slightly. 

In [ ]:
from counterfactual_xai.utils.clue.evaluation.interpret import get_BNN_uncertainties
# H_thresh_dict = {'compas':0.2, 'default_credit':0.5, 'wine':2, 'lsat':1}


art_a_thresh_dict = {'compas':0.15, 'wine':0.8, 'default_credit':0.45, 'lsat':0.68}
art_e_thresh_dict = {'compas':0.02, 'wine':0.35, 'default_credit':0.008, 'lsat':0.07}


total_stack, aleatoric_stack, epistemic_stack  = get_BNN_uncertainties(art_BNN, x_art_test,
                                           regression=regression_bools[names.index(dname)], batch_size=1024,
                                           norm_MNIST=False, flatten=False, return_probs=False, prob_BNN=True)


aleatoric_idxs = aleatoric_stack.cpu().numpy()>=art_a_thresh_dict[dname]
epistemic_idxs = epistemic_stack.cpu().numpy()>=art_e_thresh_dict[dname]

fig, ax = plt.subplots(nrows=1, ncols=2, dpi=100)
ax[0].hist(aleatoric_stack.cpu().numpy(), alpha=0.5, density=True)
ax[0].axvline(art_a_thresh_dict[dname], c='r')
ax[0].set_title('aleatoric')
ax[1].hist(epistemic_stack.cpu().numpy(), alpha=0.5, density=True)
ax[1].axvline(art_e_thresh_dict[dname], c='r')
ax[1].set_title('epistemic')
# print(np.log(2))

# x_art_test_thresh = x_art_test[]

## Get Raw GT VAEAC + BNN baselines

### VAEAC Aleatoric of orignal artificial data

In [ ]:
from counterfactual_xai.utils.clue.evaluation.interpret import evaluate_aleatoric_explanation_cat, evaluate_aleatoric_explanation_gauss

if regression_bools[names.index(dname)]:
    OG_VAEAC_aleatoric_uncert = evaluate_aleatoric_explanation_gauss(\
                                                     VAEAC, torch.Tensor(x_art_test[aleatoric_idxs]), test_dims[d_idx],
                                                      pred_sig=(not gauss_cat_vae_bools[d_idx]),
                                                             N_target_samples=500, batch_size=1024)
else:
    OG_VAEAC_aleatoric_uncert = evaluate_aleatoric_explanation_cat(VAEAC, torch.Tensor(x_art_test[aleatoric_idxs]), test_dims[d_idx],
                                                              N_target_samples=500, batch_size=1024)

print(OG_VAEAC_aleatoric_uncert.shape)
print('baseline BNN aleatoric uncert', aleatoric_stack.data.cpu().numpy()[aleatoric_idxs].mean(axis=0))


print('test GT marginal shape', OG_VAEAC_aleatoric_uncert.shape)
print('test GT mean uncert (aleatoric)', OG_VAEAC_aleatoric_uncert.mean())



### BNN error (epistemic) on original aritficial data 


In [ ]:
from counterfactual_xai.utils.clue.evaluation.interpret import evaluate_epistemic_explanation_cat, evaluate_epistemic_explanation_gauss

if regression_bools[names.index(dname)]:
    GT_test_err, GT_abs_diffs = evaluate_epistemic_explanation_gauss(art_BNN, VAEAC, torch.Tensor(x_art_test[epistemic_idxs]).cuda(), test_dims=test_dims[d_idx],
                                                                       pred_sig=(not gauss_cat_vae_bools[d_idx]), outer_batch_size=2000,
                                                                       inner_batch_size=1024, VAEAC_samples=500)
else:
    GT_test_err, GT_loglike_vec = evaluate_epistemic_explanation_cat(art_BNN, VAEAC, torch.Tensor(x_art_test[epistemic_idxs]).cuda(),
                                                                 test_dims=test_dims[d_idx], outer_batch_size=2000,
                                                                 inner_batch_size=1024, VAEAC_samples=500)


print('baseline mean BNN epistemic', epistemic_stack.data.cpu().numpy()[epistemic_idxs].mean())

print('baseline mean BNN error', GT_test_err)
if regression_bools[names.index(dname)]:
    print('baseline mean BNN other err', GT_abs_diffs.mean())
else:
    print('baseline mean BNN loglike', GT_loglike_vec.mean())



### VAEAC loglike

In [ ]:
dname

In [ ]:
from counterfactual_xai.utils.clue.evaluation.interpret import get_VAEAC_px, get_VAEAC_px_gauss_cat


if gauss_cat_vae_bools[names.index(dname)]:
    # override_y_dims doesnt use y_dims for likelihood calculation but instead uses :-override. 
    log_px_vaeac_a = get_VAEAC_px_gauss_cat(under_VAEAC_net, x_art_test[aleatoric_idxs], input_dim_vec=input_dim_vec_xy,
                                          y_dims=test_dims[d_idx], override_y_dims=1, Nsamples=1000)
else:
    log_px_vaeac_a = get_VAEAC_px(under_VAEAC_net, x_art_test[aleatoric_idxs], y_dims=test_dims[d_idx], Nsamples=10000)
    
if gauss_cat_vae_bools[names.index(dname)]:
    # override_y_dims doesnt use y_dims for likelihood calculation but instead uses :-override.
    log_px_vaeac_e = get_VAEAC_px_gauss_cat(under_VAEAC_net, x_art_test[epistemic_idxs], input_dim_vec=input_dim_vec_xy,
                                          y_dims=test_dims[d_idx], override_y_dims=1, Nsamples=1000)
else:
    log_px_vaeac_e = get_VAEAC_px(under_VAEAC_net, x_art_test[epistemic_idxs], y_dims=test_dims[d_idx], Nsamples=10000)    


print('baseline log_px_a [mean, std]', log_px_vaeac_a.mean(), log_px_vaeac_a.std())
print('baseline log_px_e [mean, std]', log_px_vaeac_e.mean(), log_px_vaeac_e.std())


## input Sensitivity

In [ ]:
sensitivity_steps = np.logspace(np.log(0.001), np.log(200), 20, base=np.e)

### Aleatoric Sensitivity RUN

In [ ]:
from counterfactual_xai.utils.clue.evaluation.interpret import input_uncertainty_step_gauss, input_uncertainty_step_cat
# from interpret.explanation_tools import input_uncertainty_step_gauss, input_uncertainty_step_cat
from counterfactual_xai.methods.clue import CLUE



sense_aleatoric_delta_H_vec = []
sense_aleatoric_delta_X_vec = []
sense_aleatoric_logpx_vec = []


for step in sensitivity_steps:

    aleatoric_coeff=1
    epistemic_coeff=0



    if regression_bools[names.index(dname)]:

        sens_x_aleatoric = input_uncertainty_step_gauss(art_BNN, art_valset, aleatoric_coeff=aleatoric_coeff,
                                              epistemic_coeff=epistemic_coeff, stepsize_perdim=-step,
                                     batch_size=1024, cuda=True, entropy=False, norm_grad=False)

    else:

        sens_x_aleatoric = input_uncertainty_step_cat(art_BNN, art_valset, aleatoric_coeff=aleatoric_coeff,
                                                      epistemic_coeff=epistemic_coeff, stepsize_perdim=-step,
                                 batch_size=1024, cuda=True, norm_MNIST=(dname=='MNIST'), flatten=False,
                                                      norm_grad=False)

    sens_x_aleatoric = sens_x_aleatoric.data.cpu().numpy()[aleatoric_idxs]
########################################################################################################

    total_uncert_BNN_sens_aleatoric, aleatoric_uncert_BNN_sens_aleatoric, epistemic_uncert_BNN_sens_aleatoric\
            = get_BNN_uncertainties(art_BNN, sens_x_aleatoric,
                                       regression=regression_bools[names.index(dname)], batch_size=1024,
                                       norm_MNIST=False, flatten=False, return_probs=False, prob_BNN=True)

    if regression_bools[names.index(dname)]:
        sens_VAEAC_aleatoric_uncert = evaluate_aleatoric_explanation_gauss(\
                                                         VAEAC, torch.Tensor(sens_x_aleatoric), test_dims[d_idx],
                                                          pred_sig=(not gauss_cat_vae_bools[d_idx]),
                                                                 N_target_samples=500, batch_size=1024)
    else:
        sens_VAEAC_aleatoric_uncert = evaluate_aleatoric_explanation_cat(VAEAC, torch.Tensor(sens_x_aleatoric),
                                                                         test_dims[d_idx],
                                                                  N_target_samples=500, batch_size=1024)


    print(sens_VAEAC_aleatoric_uncert.shape)
    # np.save(experiment_dir+'baseline_BNN_aleatoric_std.npy', OG_BNN_aleatoric_std.data.cpu().numpy())
    print('original+ sens_aleatoric BNN aleatoric uncert',aleatoric_stack.cpu().numpy()[aleatoric_idxs].mean(),\
          aleatoric_uncert_BNN_sens_aleatoric.data.cpu().numpy().mean(axis=0))


    print('original+ test sens_aleatoric marginal shape', sens_VAEAC_aleatoric_uncert.shape)
    print('original+ test sens_aleatoric mean uncert (aleatoric)', OG_VAEAC_aleatoric_uncert.mean(), sens_VAEAC_aleatoric_uncert.mean())


    sens_aleatoric_deltax = np.abs(x_art_test[aleatoric_idxs] - sens_x_aleatoric).sum(axis=1)
    deltaH = (OG_VAEAC_aleatoric_uncert - sens_VAEAC_aleatoric_uncert).cpu().numpy()
    sens_aleatoric_fm = (OG_VAEAC_aleatoric_uncert - sens_VAEAC_aleatoric_uncert).cpu().numpy() / (sens_aleatoric_deltax + 1e-6)
    print('test sens_aleatoric fm mean std (aleatoric)', np.nanmean(sens_aleatoric_fm), np.nanstd(sens_aleatoric_fm))

    #######################################


    if gauss_cat_vae_bools[names.index(dname)]:
        # override_y_dims doesnt use y_dims for likelihood calculation but instead uses :-override. 
        log_px_vaeac_sens_aleatoric = get_VAEAC_px_gauss_cat(under_VAEAC_net, sens_x_aleatoric, input_dim_vec=input_dim_vec_xy,
                                              y_dims=test_dims[d_idx], override_y_dims=1, Nsamples=1000)
    else:
        log_px_vaeac_sens_aleatoric = get_VAEAC_px(under_VAEAC_net, sens_x_aleatoric, y_dims=test_dims[d_idx],
                                                   Nsamples=1000)

    print('sens_aleatoric log_px [mean, std]', log_px_vaeac_a.mean(), log_px_vaeac_sens_aleatoric.mean(), log_px_vaeac_sens_aleatoric.std())

    sense_aleatoric_delta_H_vec.append(deltaH)
    sense_aleatoric_delta_X_vec.append(sens_aleatoric_deltax)
    sense_aleatoric_logpx_vec.append(log_px_vaeac_sens_aleatoric.data.cpu().numpy())
    
sense_aleatoric_delta_H_vec = np.stack(sense_aleatoric_delta_H_vec, axis=0)
sense_aleatoric_delta_X_vec = np.stack(sense_aleatoric_delta_X_vec, axis=0)
sense_aleatoric_logpx_vec = np.stack(sense_aleatoric_logpx_vec, axis=0)


### Epistemic Sensitivity RUN

In [ ]:
sense_epistemic_delta_err_vec = []
sense_epistemic_delta_X_vec = []
sense_epistemic_logpx_vec = []


for step in sensitivity_steps:

#     step = sensitivity_step_dict_e[dname]

    aleatoric_coeff=0
    epistemic_coeff=1


    if regression_bools[names.index(dname)]:

        sens_x_epistemic = input_uncertainty_step_gauss(art_BNN, art_valset, aleatoric_coeff=aleatoric_coeff,
                                              epistemic_coeff=epistemic_coeff, stepsize_perdim=-step,
                                     batch_size=1024, cuda=True, entropy=False, norm_grad=False)

    else:

        sens_x_epistemic = input_uncertainty_step_cat(art_BNN, art_valset, aleatoric_coeff=aleatoric_coeff,
                                                      epistemic_coeff=epistemic_coeff, stepsize_perdim=-step,
                                 batch_size=1024, cuda=True, norm_MNIST=(dname=='MNIST'), flatten=False,
                                                      norm_grad=False)


    sens_x_epistemic = sens_x_epistemic.data.cpu().numpy()[epistemic_idxs]

    #####################################################################

    total_uncert_BNN_sens_epistemic, aleatoric_uncert_BNN_sens_epistemic, epistemic_uncert_BNN_sens_epistemic\
            = get_BNN_uncertainties(art_BNN, sens_x_epistemic,
                                       regression=regression_bools[names.index(dname)], batch_size=1024,
                                       norm_MNIST=False, flatten=False, return_probs=False, prob_BNN=True)


    if regression_bools[names.index(dname)]:
        sens_test_err, sens_abs_diffs = evaluate_epistemic_explanation_gauss(art_BNN, VAEAC, torch.Tensor(sens_x_epistemic).cuda(), test_dims=test_dims[d_idx],
                                                                           pred_sig=(not gauss_cat_vae_bools[d_idx]), outer_batch_size=2000,
                                                                           inner_batch_size=1024, VAEAC_samples=500)
    else:
        sens_test_err, sens_loglike_vec = evaluate_epistemic_explanation_cat(art_BNN, VAEAC, torch.Tensor(sens_x_epistemic).cuda(),
                                                                     test_dims=test_dims[d_idx], outer_batch_size=2000,
                                                                     inner_batch_size=1024, VAEAC_samples=500)


    print('original+ sens_epistemic mean BNN epistemic', epistemic_stack.data.cpu().numpy()[epistemic_idxs].mean(), epistemic_uncert_BNN_sens_epistemic.mean())

    print('original+ sens_epistemic mean BNN error', GT_test_err,  sens_test_err)

    if regression_bools[names.index(dname)]:
        print('original+ sens_epistemic BNN other err', sens_abs_diffs.mean())
    else:
        print('original+ sens_epistemic loglike', sens_loglike_vec.mean())



    sens_epistemic_deltax = np.abs(x_art_test[epistemic_idxs] - sens_x_epistemic).sum(axis=1)
    sens_epistemic_fm = (GT_test_err - sens_test_err) / (sens_epistemic_deltax + 1e-6)
    print('test sens_epistemic fm mean std (epistemic)', np.nanmean(sens_epistemic_fm), np.nanstd(sens_epistemic_fm))


    ##########################################################################
    

    if gauss_cat_vae_bools[names.index(dname)]:
        # override_y_dims doesnt use y_dims for likelihood calculation but instead uses :-override. Maybe for unflattened?
        log_px_vaeac_sens_epistemic = get_VAEAC_px_gauss_cat(under_VAEAC_net, sens_x_epistemic, input_dim_vec=input_dim_vec_xy,
                                              y_dims=test_dims[d_idx], override_y_dims=1, Nsamples=1000)
    else:
        log_px_vaeac_sens_epistemic = get_VAEAC_px(under_VAEAC_net, sens_x_epistemic, y_dims=test_dims[d_idx],
                                                   Nsamples=1000)

    # log_px_vaeac = log_px_vaeac[log_px_vaeac>hard_min_lox_px]

    print('sens epistemic log_px [mean, std]', log_px_vaeac_e.mean(), log_px_vaeac_sens_epistemic.mean(), log_px_vaeac_sens_epistemic.std())

    sense_epistemic_delta_err_vec.append((GT_test_err - sens_test_err))
    sense_epistemic_delta_X_vec.append(sens_epistemic_deltax)
    sense_epistemic_logpx_vec.append(log_px_vaeac_sens_epistemic.data.cpu().numpy())
    
sense_epistemic_delta_err_vec = np.stack(sense_epistemic_delta_err_vec, axis=0)
sense_epistemic_delta_X_vec = np.stack(sense_epistemic_delta_X_vec, axis=0)
sense_epistemic_logpx_vec = np.stack(sense_epistemic_logpx_vec, axis=0)

## CLUE



In [ ]:
from counterfactual_xai.methods.interpretation import latent_project_gauss, latent_project_cat
from __future__ import division
import numpy as np

CLUE_lr_dic_a = {'compas':0.1, 'default_credit':0.1, 'wine':0.1, 'lsat':0.1}
CLUE_lr_dic_e = {'compas':0.1, 'default_credit':0.1, 'wine':0.1, 'lsat':0.1}

# These values change due to different statistics of artificial data

CLUE_lambda_dic_a = {'compas':2/2, 'default_credit':3, 'wine':2.5/1.5, 'lsat':1.5/4}
CLUE_lambda_dic_e = {'compas':2/4, 'default_credit':3, 'wine':2.5/2, 'lsat':1.5/6}


if regression_bools[names.index(dname)]:
    _, _, z_test, x_test, _ = \
        latent_project_gauss(art_BNN, VAE_art, dset=art_valset, batch_size=2048, cuda=cuda)
else:
    _, _, z_test, x_test, _ = \
        latent_project_cat(art_BNN, VAE_art, dset=art_valset, batch_size=2048, cuda=cuda)

    
z_init_batch_a = z_test[aleatoric_idxs]
x_init_batch_a = x_test[aleatoric_idxs]

z_init_batch_e = z_test[epistemic_idxs]
x_init_batch_e = x_test[epistemic_idxs]

CLUE_lambdas = np.logspace(np.log(0.0001), np.log(30), 30, base=np.e)


### Run Aleatoric CLUE

In [ ]:
class Ln_distance(nn.Module):
    """If dims is None Compute across all dimensions except first"""
    def __init__(self, n, dim=None):
        super(Ln_distance, self).__init__()
        self.n = n
        self.dim = dim

    def forward(self, x, y):
        d = x - y
        if self.dim is None:
            self.dim = list(range(1, len(d.shape)))
        return torch.abs(d).pow(self.n).sum(dim=self.dim).pow(1./float(self.n))


In [ ]:
torch.cuda.empty_cache()

dist = Ln_distance(n=1, dim=(1))
x_dim = x_init_batch_a.reshape(x_init_batch_a.shape[0], -1).shape[1]

lr = CLUE_lr_dic_a[dname]

aleatoric_weight = 1
epistemic_weight = 0
uncertainty_weight = 0

CLUE_aleatoric_delta_H_vec = []
CLUE_aleatoric_delta_X_vec = []
CLUE_aleatoric_logpx_vec = []


for distance_weight in (CLUE_lambdas / x_dim):

    prediction_similarity_weight = 0


    CLUE_explainer = CLUE(VAE_art, art_BNN, x_init_batch_a, uncertainty_weight=uncertainty_weight,
                          aleatoric_weight=aleatoric_weight, epistemic_weight=epistemic_weight,
                          prior_weight=0, distance_weight=distance_weight,
                     latent_L2_weight=0, prediction_similarity_weight=prediction_similarity_weight,
                     lr=lr, desired_preds=None, cond_mask=None, distance_metric=dist,
                     z_init=z_init_batch_a, norm_MNIST=False,
                     flatten_BNN=False, regression=regression_bools[names.index(dname)], cuda=True)

    torch.autograd.set_detect_anomaly(False)

    z_vec, x_vec, uncertainty_vec, epistemic_vec, aleatoric_vec, cost_vec, dist_vec = CLUE_explainer.optimise(
                                            min_steps=3, max_steps=55,
                                            n_early_stop=3)



    plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.5, hspace=None)

    x_CLUE_aleatoric = x_vec[-1]

    #################################################################

    total_uncert_BNN_CLUE_aleatoric, aleatoric_uncert_BNN_CLUE_aleatoric, epistemic_uncert_BNN_CLUE_aleatoric\
            = get_BNN_uncertainties(art_BNN, x_CLUE_aleatoric,
                                       regression=regression_bools[names.index(dname)], batch_size=1024,
                                       norm_MNIST=False, flatten=False, return_probs=False, prob_BNN=True)

    if regression_bools[names.index(dname)]:
        CLUE_VAEAC_aleatoric_uncert = evaluate_aleatoric_explanation_gauss(\
                                                         VAEAC, torch.Tensor(x_CLUE_aleatoric), test_dims[d_idx],
                                                          pred_sig=(not gauss_cat_vae_bools[d_idx]),
                                                                 N_target_samples=500, batch_size=1024)
    else:
        CLUE_VAEAC_aleatoric_uncert = evaluate_aleatoric_explanation_cat(VAEAC, torch.Tensor(x_CLUE_aleatoric), test_dims[d_idx],
                                                                  N_target_samples=500, batch_size=1024)


    print(CLUE_VAEAC_aleatoric_uncert.shape)
    # np.save(experiment_dir+'baseline_BNN_aleatoric_std.npy', OG_BNN_aleatoric_std.data.cpu().numpy())
    print('original+ CLUE_aleatoric BNN aleatoric uncert', aleatoric_stack.cpu().numpy()[aleatoric_idxs].mean(),\
          aleatoric_uncert_BNN_CLUE_aleatoric.data.cpu().numpy().mean(axis=0))

    print('original+ test CLUE_aleatoric marginal shape', CLUE_VAEAC_aleatoric_uncert.shape)
    print('original+ test CLUE_aleatoric mean uncert (aleatoric)', OG_VAEAC_aleatoric_uncert.mean(), CLUE_VAEAC_aleatoric_uncert.mean())


    CLUE_aleatoric_deltax = np.abs(x_init_batch_a - x_CLUE_aleatoric).sum(axis=1)
    CLUE_aleatoric_deltaH = (OG_VAEAC_aleatoric_uncert - CLUE_VAEAC_aleatoric_uncert).cpu().numpy() 
    CLUE_aleatoric_fm = (OG_VAEAC_aleatoric_uncert - CLUE_VAEAC_aleatoric_uncert).cpu().numpy() / (CLUE_aleatoric_deltax + 1e-6)
    print('test CLUE_aleatoric fm mean std (aleatoric)', np.nanmean(CLUE_aleatoric_fm), np.nanstd(CLUE_aleatoric_fm))


    #####################################################

    if gauss_cat_vae_bools[names.index(dname)]:
        # override_y_dims doesnt use y_dims for likelihood calculation but instead uses :-override. Maybe for unflattened?
        log_px_vaeac_CLUE_aleatoric = get_VAEAC_px_gauss_cat(under_VAEAC_net, x_CLUE_aleatoric, input_dim_vec=input_dim_vec_xy,
                                              y_dims=test_dims[d_idx], override_y_dims=1, Nsamples=1000)
    else:
        log_px_vaeac_CLUE_aleatoric = get_VAEAC_px(under_VAEAC_net, x_CLUE_aleatoric, y_dims=test_dims[d_idx],
                                                   Nsamples=1000)

    # log_px_vaeac = log_px_vaeac[log_px_vaeac>hard_min_lox_px]

    print('original+ CLUE_aleatoric log_px [mean, std]', log_px_vaeac_a.mean(), log_px_vaeac_CLUE_aleatoric.mean(), log_px_vaeac_CLUE_aleatoric.std())

    
    CLUE_aleatoric_delta_H_vec.append(CLUE_aleatoric_deltaH)
    CLUE_aleatoric_delta_X_vec.append(CLUE_aleatoric_deltax)
    CLUE_aleatoric_logpx_vec.append(log_px_vaeac_CLUE_aleatoric.data.cpu().numpy())
    
CLUE_aleatoric_delta_H_vec = np.stack(CLUE_aleatoric_delta_H_vec, axis=0)
CLUE_aleatoric_delta_X_vec = np.stack(CLUE_aleatoric_delta_X_vec, axis=0)
CLUE_aleatoric_logpx_vec = np.stack(CLUE_aleatoric_logpx_vec, axis=0)


### Run epistemic CLUE

In [ ]:
torch.cuda.empty_cache()

dist = Ln_distance(n=1, dim=(1))
x_dim = x_init_batch_e.reshape(x_init_batch_e.shape[0], -1).shape[1]

lr = CLUE_lr_dic_e[dname]

aleatoric_weight = 0
epistemic_weight = 1
uncertainty_weight = 0

CLUE_epistemic_delta_err_vec = []
CLUE_epistemic_delta_X_vec = []
CLUE_epistemic_logpx_vec = []


for distance_weight in (CLUE_lambdas / x_dim):
    prediction_similarity_weight = 0


    CLUE_explainer = CLUE(VAE_art, art_BNN, x_init_batch_e, uncertainty_weight=uncertainty_weight, aleatoric_weight=aleatoric_weight, epistemic_weight=epistemic_weight,
                          prior_weight=0, distance_weight=distance_weight,
                     latent_L2_weight=0, prediction_similarity_weight=prediction_similarity_weight,
                     lr=lr, desired_preds=None, cond_mask=None, distance_metric=dist,
                     z_init=z_init_batch_e, norm_MNIST=False,
                     flatten_BNN=False, regression=regression_bools[names.index(dname)], cuda=True)

    torch.autograd.set_detect_anomaly(False)

    # clue_instance.optimizer = SGD(self.trainable_params, lr=lr, momentum=0.5, nesterov=True)
    z_vec, x_vec, uncertainty_vec, epistemic_vec, aleatoric_vec, cost_vec, dist_vec = CLUE_explainer.optimise(
                                            min_steps=3, max_steps=65,
                                            n_early_stop=3)

    fig, axes = plt.subplots(1, 3, dpi=130)
    axes[0].plot(cost_vec.mean(axis=1))
    axes[0].set_title('mean Cost')
    axes[0].set_xlabel('iterations')

    axes[1].plot(uncertainty_vec.mean(axis=1))
    axes[1].set_title('mean Total Entropy')
    axes[1].set_xlabel('iterations')

    axes[2].plot(dist_vec.mean(axis=1))
    axes[2].set_title('mean Ln Cost')
    axes[2].set_xlabel('iterations')

    plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.5, hspace=None)

    x_CLUE_epistemic = x_vec[-1]
    
    ############################################################################
    total_uncert_BNN_CLUE_epistemic, aleatoric_uncert_BNN_CLUE_epistemic, epistemic_uncert_BNN_CLUE_epistemic\
            = get_BNN_uncertainties(art_BNN, x_CLUE_epistemic,
                                       regression=regression_bools[names.index(dname)], batch_size=1024,
                                       norm_MNIST=False, flatten=False, return_probs=False, prob_BNN=True)


    if regression_bools[names.index(dname)]:
        CLUE_test_err, CLUE_abs_diffs = evaluate_epistemic_explanation_gauss(art_BNN, VAEAC, torch.Tensor(x_CLUE_epistemic).cuda(), test_dims=test_dims[d_idx],
                                                                           pred_sig=(not gauss_cat_vae_bools[d_idx]), outer_batch_size=2000,
                                                                           inner_batch_size=1024, VAEAC_samples=500)
    else:
        CLUE_test_err, CLUE_loglike_vec = evaluate_epistemic_explanation_cat(art_BNN, VAEAC, torch.Tensor(x_CLUE_epistemic).cuda(),
                                                                     test_dims=test_dims[d_idx], outer_batch_size=2000,
                                                                     inner_batch_size=1024, VAEAC_samples=500)


    print('original+ CLUE_epistemic mean BNN epistemic', epistemic_stack.cpu().numpy()[epistemic_idxs].mean(),\
          epistemic_uncert_BNN_CLUE_epistemic.mean())

    print('original+ CLUE_epistemic mean BNN error', GT_test_err, CLUE_test_err)

    if regression_bools[names.index(dname)]:
        print('original+ CLUE_epistemic BNN other err', CLUE_abs_diffs.mean())
    else:
        print('original+ CLUE_epistemic loglike', CLUE_loglike_vec.mean())


    CLUE_epistemic_deltax = np.abs(x_init_batch_e - x_CLUE_epistemic).sum(axis=1)
    CLUE_epistemic_fm = (GT_test_err - CLUE_test_err) / (CLUE_epistemic_deltax + 1e-6)
    print('test CLUE_epistemic fm mean std (epistemic)', np.nanmean(CLUE_epistemic_fm), np.nanstd(CLUE_epistemic_fm))

    ###########################################################

    
    if gauss_cat_vae_bools[names.index(dname)]:
        # override_y_dims doesnt use y_dims for likelihood calculation but instead uses :-override. Maybe for unflattened?
        log_px_vaeac_CLUE_epistemic = get_VAEAC_px_gauss_cat(under_VAEAC_net, x_CLUE_epistemic, input_dim_vec=input_dim_vec_xy,
                                              y_dims=test_dims[d_idx], override_y_dims=1, Nsamples=1000)
    else:
        log_px_vaeac_CLUE_epistemic = get_VAEAC_px(under_VAEAC_net, x_CLUE_epistemic,
                                                   y_dims=test_dims[d_idx], Nsamples=1000)

    # log_px_vaeac = log_px_vaeac[log_px_vaeac>hard_min_lox_px]

    print('original+ epistemic log_px [mean, std]', log_px_vaeac_e.mean(), log_px_vaeac_CLUE_epistemic.mean(), log_px_vaeac_CLUE_epistemic.std())

    
    CLUE_epistemic_delta_err_vec.append((GT_test_err - CLUE_test_err))
    CLUE_epistemic_delta_X_vec.append(CLUE_epistemic_deltax)
    CLUE_epistemic_logpx_vec.append(log_px_vaeac_CLUE_epistemic.data.cpu().numpy())
    
CLUE_epistemic_delta_err_vec = np.stack(CLUE_epistemic_delta_err_vec, axis=0)
CLUE_epistemic_delta_X_vec = np.stack(CLUE_epistemic_delta_X_vec, axis=0)
CLUE_epistemic_logpx_vec = np.stack(CLUE_epistemic_logpx_vec, axis=0)

## U-FIDO

In [ ]:


fido_lambdas = np.logspace(np.log(0.00001), np.log(100), 30, base=np.e)



### Run FIDO aleatoric

In [ ]:
from counterfactual_xai.utils.clue.evaluation.interpret import generate_ind_batch, mask_explainer

batch_size = 3000


FIDO_aleatoric_delta_H_vec = []
FIDO_aleatoric_delta_X_vec = []
FIDO_aleatoric_logpx_vec = []
for L1w in fido_lambdas:

    FIDO_explanations = []
    FIDO_masks = []
    
    aleatoric_coeff = 1
    epistemic_coeff = 0

    aux_loader = generate_ind_batch(x_init_batch_a.shape[0], batch_size, random=False, roundup=True)
    for idxs in aux_loader:

        explainer, loss_vec, aleatoric_vec, epistemic_vec = \
                mask_explainer.train_mask(torch.Tensor(x_init_batch_a[idxs]), art_BNN, VAEAC_art, aleatoric_coeff=aleatoric_coeff,
                                          epistemic_coeff=epistemic_coeff, L1w=L1w, N_epochs=30,
                              mask_samples=20, mask_samples2=10, flatten_ims=False,
                                          test_dims=None, cat=(not regression_bools[names.index(dname)])) 

        explanation, mask = explainer.mask_inpaint(torch.Tensor(x_init_batch_a[idxs]), VAEAC_art,
                                                   flatten_ims=False, test_dims=None,
                                                   cat=(not regression_bools[names.index(dname)]))

        FIDO_explanations.append(explanation)
        FIDO_masks.append(mask)

    FIDO_explanations = torch.cat(FIDO_explanations, dim=0).data.cpu()
    FIDO_masks = torch.cat(FIDO_masks, dim=0).data.cpu()

    print(FIDO_explanations.shape)

    explainer = None
    torch.cuda.empty_cache()

    x_FIDO_aleatoric  = FIDO_explanations.numpy()

    ###############################################

    total_uncert_BNN_FIDO_aleatoric, aleatoric_uncert_BNN_FIDO_aleatoric, epistemic_uncert_BNN_FIDO_aleatoric\
            = get_BNN_uncertainties(art_BNN, x_FIDO_aleatoric,
           regression=regression_bools[names.index(dname)], batch_size=1024,
           norm_MNIST=False, flatten=False, return_probs=False, prob_BNN=True)

    if regression_bools[names.index(dname)]:
        FIDO_VAEAC_aleatoric_uncert = evaluate_aleatoric_explanation_gauss(\
                                                         VAEAC, torch.Tensor(x_FIDO_aleatoric), test_dims[d_idx],
                                                          pred_sig=(not gauss_cat_vae_bools[d_idx]),
                                                                 N_target_samples=500, batch_size=1024)
    else:
        FIDO_VAEAC_aleatoric_uncert = evaluate_aleatoric_explanation_cat(VAEAC, torch.Tensor(x_FIDO_aleatoric),
                                                                     test_dims[d_idx],
                                                                  N_target_samples=500, batch_size=1024)



    # np.save(experiment_dir+'baseline_BNN_aleatoric_std.npy', OG_BNN_aleatoric_std.data.cpu().numpy())
    print('original+ FIDO_aleatoric BNN aleatoric uncert', aleatoric_stack.cpu().numpy()[aleatoric_idxs].mean(),
                              aleatoric_uncert_BNN_FIDO_aleatoric.data.cpu().numpy().mean(axis=0))


    print('original+ test FIDO_aleatoric marginal shape', FIDO_VAEAC_aleatoric_uncert.shape)
    print('original+ test FIDO_aleatoric mean uncert (aleatoric)', OG_VAEAC_aleatoric_uncert.mean(), FIDO_VAEAC_aleatoric_uncert.mean())


    FIDO_aleatoric_deltax = np.abs(x_init_batch_a - x_FIDO_aleatoric).sum(axis=1)
    FIDO_aleatoric_deltaH = (OG_VAEAC_aleatoric_uncert - FIDO_VAEAC_aleatoric_uncert).cpu().numpy()
    FIDO_aleatoric_fm = (OG_VAEAC_aleatoric_uncert - FIDO_VAEAC_aleatoric_uncert).cpu().numpy() / (FIDO_aleatoric_deltax + 1e-6)
    print('test FIDO_aleatoric fm mean std (aleatoric)', np.nanmean(FIDO_aleatoric_fm), np.nanstd(FIDO_aleatoric_fm))

    #############################################################
    
    # We give complete input dim vec so that the GT VAEAC can do its thing
    # Why is override y dims not implemented by using test_dims? -> because it unites categoricals

    if gauss_cat_vae_bools[names.index(dname)]:
        # override_y_dims doesnt use y_dims for likelihood calculation but instead uses :-override. Maybe for unflattened?
        log_px_vaeac_FIDO_aleatoric = get_VAEAC_px_gauss_cat(under_VAEAC_net, x_FIDO_aleatoric, input_dim_vec=input_dim_vec_xy,
                                              y_dims=test_dims[d_idx], override_y_dims=1, Nsamples=1000)
    else:
        log_px_vaeac_FIDO_aleatoric = get_VAEAC_px(under_VAEAC_net, x_FIDO_aleatoric, y_dims=test_dims[d_idx], Nsamples=10000)

    print('original+ FIDO_aleatoric log_px [mean, std]', log_px_vaeac_a.mean(), log_px_vaeac_FIDO_aleatoric.mean(), log_px_vaeac_FIDO_aleatoric.std())

    
    FIDO_aleatoric_delta_H_vec.append(FIDO_aleatoric_deltaH)
    FIDO_aleatoric_delta_X_vec.append(FIDO_aleatoric_deltax)
    FIDO_aleatoric_logpx_vec.append(log_px_vaeac_FIDO_aleatoric.data.cpu().numpy())
    
FIDO_aleatoric_delta_H_vec = np.stack(FIDO_aleatoric_delta_H_vec, axis=0)
FIDO_aleatoric_delta_X_vec = np.stack(FIDO_aleatoric_delta_X_vec, axis=0)
FIDO_aleatoric_logpx_vec = np.stack(FIDO_aleatoric_logpx_vec, axis=0)
    


### Run FIDO epistemic 

In [ ]:
batch_size = 3000

FIDO_epistemic_delta_err_vec = []
FIDO_epistemic_delta_X_vec = []
FIDO_epistemic_logpx_vec = []
for L1w in fido_lambdas:

    FIDO_explanations = []
    FIDO_masks = []

    aleatoric_coeff = 0
    epistemic_coeff = 1

    aux_loader = generate_ind_batch(x_init_batch_e.shape[0], batch_size, random=False, roundup=True)
    for idxs in aux_loader:

        explainer, loss_vec, aleatoric_vec, epistemic_vec = \
                mask_explainer.train_mask(torch.Tensor(x_init_batch_e[idxs]), art_BNN, VAEAC_art, aleatoric_coeff=aleatoric_coeff,
                                          epistemic_coeff=epistemic_coeff, L1w=L1w, N_epochs=30,
                              mask_samples=20, mask_samples2=10, flatten_ims=False,
                                          test_dims=None, cat=(not regression_bools[names.index(dname)])) 

        explanation, mask = explainer.mask_inpaint(torch.Tensor(x_init_batch_e[idxs]), VAEAC_art,
                                                   flatten_ims=False, test_dims=None,
                                                   cat=(not regression_bools[names.index(dname)]))

        FIDO_explanations.append(explanation)
        FIDO_masks.append(mask)

    FIDO_explanations = torch.cat(FIDO_explanations, dim=0).data.cpu()
    FIDO_masks = torch.cat(FIDO_masks, dim=0).data.cpu()

    print(FIDO_explanations.shape)

    explainer = None
    torch.cuda.empty_cache()

    x_FIDO_epistemic  = FIDO_explanations.numpy()

    ########################################################################
    
    total_uncert_BNN_FIDO_epistemic, aleatoric_uncert_BNN_FIDO_epistemic, epistemic_uncert_BNN_FIDO_epistemic\
            = get_BNN_uncertainties(art_BNN, x_FIDO_epistemic,
           regression=regression_bools[names.index(dname)], batch_size=1024,
           norm_MNIST=False, flatten=False, return_probs=False, prob_BNN=True)

    if regression_bools[names.index(dname)]:
        FIDO_test_err, FIDO_abs_diffs = evaluate_epistemic_explanation_gauss(art_BNN, VAEAC,
                                         torch.Tensor(x_FIDO_epistemic).cuda(), test_dims=test_dims[d_idx],
                                       pred_sig=(not gauss_cat_vae_bools[d_idx]), outer_batch_size=2000,
                                       inner_batch_size=1024, VAEAC_samples=500)
    else:
        FIDO_test_err, FIDO_loglike_vec = evaluate_epistemic_explanation_cat(art_BNN, VAEAC,
                                                         torch.Tensor(x_FIDO_epistemic).cuda(),
                                                         test_dims=test_dims[d_idx], outer_batch_size=2000,
                                                         inner_batch_size=1024, VAEAC_samples=500)


    print('original+ FIDO_epistemic mean BNN epistemic', epistemic_stack.cpu().numpy()[epistemic_idxs].mean(),
          epistemic_uncert_BNN_FIDO_epistemic.mean())

    print('original+ FIDO_epistemic mean BNN error', GT_test_err, FIDO_test_err)

    if regression_bools[names.index(dname)]:
        print('original+ FIDO_epistemic BNN other err', FIDO_abs_diffs.mean())
    else:
        print('original+ FIDO_epistemic loglike', FIDO_loglike_vec.mean())


    FIDO_epistemic_deltax = np.abs(x_init_batch_e - x_FIDO_epistemic).sum(axis=1)
    FIDO_epistemic_fm = (GT_test_err - FIDO_test_err) / (FIDO_epistemic_deltax + 1e-6)
    print('test FIDO_epistemic fm mean std (epistemic)', np.nanmean(FIDO_epistemic_fm), np.nanstd(FIDO_epistemic_fm))


    ########################################################################


    if gauss_cat_vae_bools[names.index(dname)]:
        # override_y_dims doesnt use y_dims for likelihood calculation but instead uses :-override. Maybe for unflattened?
        log_px_vaeac_FIDO_epistemic = get_VAEAC_px_gauss_cat(under_VAEAC_net, x_FIDO_epistemic, input_dim_vec=input_dim_vec_xy,
                                              y_dims=test_dims[d_idx], override_y_dims=1, Nsamples=1000)
    else:
        log_px_vaeac_FIDO_epistemic = get_VAEAC_px(under_VAEAC_net, x_FIDO_epistemic, y_dims=test_dims[d_idx], Nsamples=10000)

    print('original+ FIDO_epistemic log_px [mean mean, std]', log_px_vaeac_e.mean(), log_px_vaeac_FIDO_epistemic.mean(), log_px_vaeac_FIDO_epistemic.std())

    FIDO_epistemic_delta_err_vec.append((GT_test_err - FIDO_test_err))
    FIDO_epistemic_delta_X_vec.append(FIDO_epistemic_deltax)
    FIDO_epistemic_logpx_vec.append(log_px_vaeac_FIDO_epistemic.data.cpu().numpy())
    
FIDO_epistemic_delta_err_vec = np.stack(FIDO_epistemic_delta_err_vec, axis=0)
FIDO_epistemic_delta_X_vec = np.stack(FIDO_epistemic_delta_X_vec, axis=0)
FIDO_epistemic_logpx_vec = np.stack(FIDO_epistemic_logpx_vec, axis=0)


In [ ]:
def find_extrema(array_list, use_max=True):
    
    joint = np.concatenate(array_list, axis=0)
    if use_max:
        return joint.max()
    else:
        return joint.min()
    

In [ ]:

plt.figure(dpi=100)
plt.plot(sense_epistemic_delta_X_vec.mean(axis=1), -sense_epistemic_delta_err_vec, '--.', c='b')
plt.plot(CLUE_epistemic_delta_X_vec.mean(axis=1), -CLUE_epistemic_delta_err_vec, '--.', c='r')
plt.plot(FIDO_epistemic_delta_X_vec.mean(axis=1), -FIDO_epistemic_delta_err_vec, '--.', c='g')
plt.title(dname + ' epistemic')
plt.legend(['sensitivity', 'CLUE', 'U-FIDO'])
plt.xlabel('L1 change in X')
plt.ylabel('chnage in err')
# plt.ylim([-0.1, 0.3])
# plt.xlim([0, 10])
# ax = plt.gca()
# ax.invert_xaxis()

plt.figure(dpi=100)
plt.plot(sense_epistemic_logpx_vec.mean(axis=1), -sense_epistemic_delta_err_vec, '--.', c='b')
plt.plot(CLUE_epistemic_logpx_vec.mean(axis=1), -CLUE_epistemic_delta_err_vec, '--.', c='r')
plt.plot(FIDO_epistemic_logpx_vec.mean(axis=1), -FIDO_epistemic_delta_err_vec, '--.', c='g')
ax = plt.gca()
ax.axvline(x=log_px_vaeac_e.mean())
plt.title(dname + ' epistemic')
plt.legend(['sensitivity', 'CLUE', 'U-FIDO'])
plt.xlabel('log p(x)')
plt.ylabel('change in err')
# plt.ylim([-0.1, 0.3])
# plt.xlim([log_px_vaeac_e.mean().item()-20, log_px_vaeac_e.mean().item()+5])
ax.invert_xaxis()


plt.figure(dpi=100)
plt.plot(sense_aleatoric_delta_X_vec.mean(axis=1), -sense_aleatoric_delta_H_vec.mean(axis=1), '--.', c='b')
plt.plot(CLUE_aleatoric_delta_X_vec.mean(axis=1), -CLUE_aleatoric_delta_H_vec.mean(axis=1), '--.', c='r')
plt.plot(FIDO_aleatoric_delta_X_vec.mean(axis=1), -FIDO_aleatoric_delta_H_vec.mean(axis=1), '--.', c='g')
plt.title(dname + ' Aleatoric')
plt.legend(['sensitivity', 'CLUE', 'U-FIDO'])
plt.xlabel('L1 change in X')
plt.ylabel('change in H')
# plt.ylim([-0.1, 0.3])
# plt.xlim([0, 10])
# ax = plt.gca()
# ax.invert_xaxis()

plt.figure(dpi=100)
plt.plot(sense_aleatoric_logpx_vec.mean(axis=1), -sense_aleatoric_delta_H_vec.mean(axis=1), '--.', c='b')
plt.plot(CLUE_aleatoric_logpx_vec.mean(axis=1), -CLUE_aleatoric_delta_H_vec.mean(axis=1), '--.', c='r')
plt.plot(FIDO_aleatoric_logpx_vec.mean(axis=1), -FIDO_aleatoric_delta_H_vec.mean(axis=1), '--.', c='g')
ax = plt.gca()
ax.axvline(x=log_px_vaeac_a.mean())

plt.title(dname + ' Aleatoric')
plt.legend(['sensitivity', 'CLUE', 'U-FIDO'])
plt.xlabel('log p(x)')
plt.ylabel('change in H')
# plt.xlim([log_px_vaeac_a.mean().item()-20, log_px_vaeac_a.mean().item()+5])
# plt.ylim([-0.1, 0.3])
ax.invert_xaxis()
# plt.ylim([-0.1, 0.3])